### Copyright 2022-2026 Crown Copyright

```
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
```

## Setup Sleeper Client
Set up the logger for the notebook and create a Sleeper Client that is connected to a Sleeper instance.

Prerequisites:
- Sleeper is already deployed
- AWS credentials and region are configured
- The sleeper Python package is installed

In [ ]:
import logging

from sleeper import SleeperClient, enable_logging
from sleeper.exceptions import SleeperApiError
from sleeper.rest.table import AddTableResponse, TableSchema

logging.basicConfig(
    level=logging.INFO,
    format="[NOTEBOOK] %(asctime)s %(levelname)s %(message)s",
    force=True,
)

jupyter_logger = logging.getLogger(__name__)

jupyter_logger.info("Starting notebook")

# Enable Sleeper Client logging (switch to DEBUG only when troubleshooting)
enable_logging(logging.INFO)

# TODO: Replace with your deployed values
table_name = "my-table"
instance_id = "instance-123"

sleeper_client = SleeperClient(instance_id)

## Table setup
Create the schema and split points for the table.


In [ ]:
row = [{"name": "key", "type": "StringType"}]
sort = [{"name": "timestamp", "type": "LongType"}]
value = [{"name": "value", "type": "StringType"}]
schema = TableSchema(rowKeyFields=row, sortKeyFields=sort, valueFields=value)

split = [
    "afhfmhqinh",
    "clhxnatvkv",
    "fbwxgcgrnp",
    "inpcqimwja",
    "ovhlzcqpvx",
    "paoyyzrddz",
    "tmwibijtqs",
    "zusqargwxz",
]

jupyter_logger.info(f"Schema: {schema}")

## Create table
Create the Sleeper table using the Sleeper Client. This makes a REST POST HTTP call to the AWS API Gateway.

This example calls the same API twice with the same request to show what happens if the table already exists.

The first call will succeed, creating the table. The second call will raise the `SleeperApiError` exception because the table already exists.

In [ ]:
for i in range(1, 3):
    jupyter_logger.info(f"Attempt {i} to call add_table")

    try:
        response: AddTableResponse = sleeper_client.add_table(table_name=table_name, schema=schema, split_points=split)
        jupyter_logger.info(f"Created table {response.tableName} with id: {response.tableId}")
    except SleeperApiError as e:
        jupyter_logger.error(e)